# 05 · Fase 1: modelo del efecto propio y de los efectos dentro de la categoría

**Qué hace este notebook:** estima el modelo de la Fase 1 de `metodologia.tex` para las **seis categorías objetivo sin programa semanal** de la tienda 44: BEVERAGES, GROCERY I, CLEANING, DAIRY, BREAD/BAKERY y LIQUOR,WINE,BEER (D‑26, D‑28, D‑30).

**El modelo.** Para cada categoría $j$ se estima, con sus SKUs $i$ y los días $t$:

$$
\mathbb{E}[y_{it}] = \exp\Big(
  \alpha_i + \gamma_{\text{dow}(t)} + \tau_{m(t)} + \mathbf{h}_t'\boldsymbol{\theta}
  + \sum_{b\in\mathcal{B}} \beta_b\, D^{(b)}_{it}
  + \sum_{b\in\mathcal{B}} \psi^{\text{cl}}_b\, S^{\text{cl}(b)}_{it}
  + \sum_{b\in\mathcal{B}} \psi^{\text{cat}}_b\, S^{\text{cat}(b)}_{it}
\Big), \qquad \mathcal{B}=\{0,\ \ell_{1\text{-}3},\ \ell_{4\text{-}6}\}
$$

| Símbolo | Columna | Qué es |
|---|---|---|
| $y_{it}$ | `unit_sales_ppml` | ventas del SKU $i$ el día $t$ (con ceros; devoluciones llevadas a 0) |
| $\alpha_i$, $\gamma_{\text{dow}}$, $\tau_m$ | `item_nbr`, `dow`, `mes` | efectos fijos de SKU, día de la semana y año‑mes (D‑16) |
| $\mathbf{h}_t$ | `dia_libre`, `adicional`, `dia_laboral_recuperado`, `evento_comercial`, `evento_otro`, `dia_pago`, `post_pago` | calendario de la tienda 44 (D‑10, D‑13) |
| $D_{it}$ | `onpromotion` y sus bloques `_l1_3`, `_l4_6` | promoción propia: el día y el promedio de los días 1–3 y 4–6 anteriores (D‑14) |
| $S^{\text{cl}}_{it}$ | `promo_clase_otros` y bloques | proporción de los otros SKUs de la misma clase en promoción |
| $S^{\text{cat}}_{it}$ | `promo_cat_otras_clases` y bloques | proporción de los SKUs de las otras clases de la categoría en promoción |

**Estimador:** PPML (regresión de Poisson por pseudo máxima verosimilitud) con efectos fijos, una ecuación por categoría (D‑08), con errores estándar agrupados por SKU y por semana (D‑15). Un coeficiente $\beta$ se lee como un cambio de $100\,(e^{\beta}-1)\,\%$ en la venta esperada cuando la variable sube en una unidad.

## Paso 1. Librerías y parámetros

- `pandas` y `numpy`: leer y manejar las tablas.
- `pyfixest`: estimar PPML con efectos fijos y errores agrupados (`fepois`).
- `scipy.stats`: valores p e intervalos con la distribución $t$.

Todos los elementos del modelo (categorías, regresores, controles, efectos fijos y agrupamiento) se definen aquí, en un solo lugar.

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import pyfixest as pf
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

# D-26, D-28, D-30: las seis categorías objetivo sin programa semanal
categorias = ["BEVERAGES", "GROCERY I", "CLEANING", "DAIRY", "BREAD/BAKERY", "LIQUOR,WINE,BEER"]

# D-14: promoción propia, de la clase y del resto de la categoría, cada una con sus bloques de rezagos
regresores_promocion = [
    "onpromotion", "onpromotion_l1_3", "onpromotion_l4_6",
    "promo_clase_otros", "promo_clase_otros_l1_3", "promo_clase_otros_l4_6",
    "promo_cat_otras_clases", "promo_cat_otras_clases_l1_3", "promo_cat_otras_clases_l4_6",
]

# D-10 y D-13: controles de calendario
controles_calendario = [
    "dia_libre", "adicional", "dia_laboral_recuperado", "evento_comercial", "evento_otro", "dia_pago", "post_pago",
]

variable_dependiente = "unit_sales_ppml"   # D-08, D-20
efectos_fijos = "item_nbr + dow + mes"     # D-16
agrupamiento = "item_nbr + semana"         # D-15

# Rutas (relativas a version_limpia)
ruta_procesados = os.path.join("..", "data", "procesados")
ruta_tablas = os.path.join("resultados", "tablas")
ruta_figuras = os.path.join("resultados", "figuras")
os.makedirs(ruta_tablas, exist_ok=True)
os.makedirs(ruta_figuras, exist_ok=True)

## Paso 2. Cargar los datos

Se leen tres archivos que ya produjeron los notebooks `01` a `03` (no se modifican):

- `panel_t44_sku_dia.parquet`: una fila por SKU y día, con las ventas y la promoción propia.
- `exposiciones_t44_sku_dia.parquet`: las mismas filas, con los rezagos y la promoción de la clase y de la categoría.
- `calendario_t44.csv`: una fila por día, con feriados, días de pago, ventana del terremoto, `dow`, `mes` y `semana`.

Las fechas se llevan al mismo formato en las tres tablas para que las uniones del paso siguiente no pierdan filas.

In [2]:
panel = pd.read_parquet(os.path.join(ruta_procesados, "panel_t44_sku_dia.parquet"))
exposiciones = pd.read_parquet(os.path.join(ruta_procesados, "exposiciones_t44_sku_dia.parquet"))
calendario = pd.read_csv(os.path.join(ruta_procesados, "calendario_t44.csv"), parse_dates=["date"])

# Mismo formato de fecha en las tres tablas
for tabla in (panel, exposiciones, calendario):
    tabla["date"] = tabla["date"].astype("datetime64[ns]")

print(f"panel:        {panel.shape[0]:,} filas")
print(f"exposiciones: {exposiciones.shape[0]:,} filas")
print(f"calendario:   {calendario.shape[0]:,} días")

panel:        1,935,758 filas
exposiciones: 1,935,758 filas
calendario:   590 días


**Salida.** El panel y las exposiciones tienen el mismo número de filas (1.935.758 SKU‑días de toda la tienda) y el calendario tiene los 590 días con operación.

## Paso 3. Armar la base de estimación

Siguiendo `INSTRUCCIONES.md` §6:

1. Quedarse con las filas de las seis categorías.
2. Unir las exposiciones por SKU y fecha, y el calendario por fecha (se verifica que no se dupliquen ni pierdan filas).
3. Excluir la ventana del terremoto, del 16‑04 al 31‑05‑2016 (D‑09).
4. Excluir las filas con alguna variable del modelo vacía: los bloques de rezagos que tocan el inicio del período o la ventana del terremoto, y los SKUs sin otros SKUs activos en su clase ese día.

Estas seis categorías no tienen días de programa semanal, así que no hay que excluirlos (se verifica).

In [3]:
# 1. Seis categorías
base = panel.loc[panel["family"].isin(categorias)].copy()
base["family"] = base["family"].astype(str)
filas_categorias = len(base)

# 2. Unir exposiciones (1 a 1) y calendario (muchos a 1)
base = base.merge(exposiciones, on=["item_nbr", "date"], how="left", validate="1:1")
base = base.merge(calendario, on="date", how="left", validate="m:1")
assert len(base) == filas_categorias, "una unión cambió el número de filas"
assert not base["dia_programa"].any(), "hay días de programa en estas categorías"

# 3. Excluir la ventana del terremoto (D-09)
base = base.loc[~base["ventana_terremoto"]]

# 4. Excluir filas con alguna variable del modelo vacía
variables_modelo = [variable_dependiente] + regresores_promocion + controles_calendario
base = base.dropna(subset=variables_modelo)

# Indicadores como 0/1 para la fórmula
for columna in ["onpromotion"] + controles_calendario:
    base[columna] = base[columna].astype("int8")

resumen_base = (
    base.groupby("family")
    .agg(skus=("item_nbr", "nunique"), sku_dias=("item_nbr", "size"), dias=("date", "nunique"))
    .reindex(categorias)
)
print(f"Filas de las seis categorías: {filas_categorias:,} -> base de estimación: {len(base):,}")
print(resumen_base.to_string())
assert not base.duplicated(["item_nbr", "date"]).any(), "hay más de una fila por SKU-día"

Filas de las seis categorías: 1,338,869 -> base de estimación: 1,198,613
                  skus  sku_dias  dias
family                                
BEVERAGES          576    258915   532
GROCERY I         1153    556533   532
CLEANING           396    197454   532
DAIRY              226    109779   532
BREAD/BAKERY       105     54466   532
LIQUOR,WINE,BEER    49     21466   532


**Salida.** De 1.338.869 SKU‑días de las seis categorías quedan 1.198.613 en la base de estimación. Cada categoría usa **532 días**: los 590 días con operación menos los 46 de la ventana del terremoto y 12 días cuyos bloques de rezago no se pueden calcular (6 al inicio del período y 6 justo después del terremoto). La columna `skus` muestra cuántos SKUs entran a cada modelo; en LIQUOR,WINE,BEER son 49 de 60, porque varios de sus SKUs están solos en su clase y no tienen "otros SKUs de la clase" con los cuales calcular $S^{\text{cl}}$.

## Paso 4. Estimar el modelo

Se estima una ecuación por categoría con `pf.fepois`. La fórmula tiene la forma `y ~ regresores | efectos fijos`: lo que va después de la barra se absorbe como efecto fijo (no se reporta). `vcov={"CRV1": "item_nbr + semana"}` pide errores estándar agrupados en dos vías, por SKU y por semana.

`pyfixest` puede mostrar dos avisos:

- `singleton fixed effect(s) dropped`: descartó SKUs con una sola observación, que no aportan información.
- `invalid value encountered in sqrt`: alguna varianza de un **control de calendario** salió negativa (pasa con controles que marcan solo uno o dos días, como `dia_laboral_recuperado` o `evento_otro`). No afecta a los coeficientes de promoción: todos tienen error estándar. Como se repetiría cada vez que se piden los errores estándar, después de esta celda ya no se vuelve a mostrar.

In [4]:
formula = f"{variable_dependiente} ~ {' + '.join(regresores_promocion + controles_calendario)} | {efectos_fijos}"
print(formula, "\n")

modelos = {}
for categoria in categorias:
    datos_j = base.loc[base["family"] == categoria]
    modelos[categoria] = pf.fepois(formula, data=datos_j, vcov={"CRV1": agrupamiento})
    print(f"{categoria:17s} observaciones: {modelos[categoria]._N:>7,}   convergió: {modelos[categoria]._convergence}")

assert all(modelo._convergence for modelo in modelos.values()), "algún modelo no convergió"

# El aviso de varianza negativa ya se mostró arriba y se explica en el texto: no se repite en las celdas siguientes
warnings.filterwarnings("ignore", message="invalid value encountered in sqrt")

unit_sales_ppml ~ onpromotion + onpromotion_l1_3 + onpromotion_l4_6 + promo_clase_otros + promo_clase_otros_l1_3 + promo_clase_otros_l4_6 + promo_cat_otras_clases + promo_cat_otras_clases_l1_3 + promo_cat_otras_clases_l4_6 + dia_libre + adicional + dia_laboral_recuperado + evento_comercial + evento_otro + dia_pago + post_pago | item_nbr + dow + mes 



/usr/local/lib/python3.13/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 1 singleton fixed effect(s) dropped from the model.
  warnings.warn(


BEVERAGES         observaciones: 258,914   convergió: True


/usr/local/lib/python3.13/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 2 singleton fixed effect(s) dropped from the model.
  warnings.warn(


GROCERY I         observaciones: 556,531   convergió: True


...

DAIRY             observaciones: 109,779   convergió: True


/usr/local/lib/python3.13/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


BREAD/BAKERY      observaciones:  54,466   convergió: True


LIQUOR,WINE,BEER  observaciones:  21,466   convergió: True


/usr/local/lib/python3.13/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


**Salida.** Los seis modelos convergen. Las observaciones usadas coinciden con la base del Paso 3, salvo 1 fila en BEVERAGES y 2 en GROCERY I que `pyfixest` descarta por ser *singletons*.

## Paso 5. Coeficientes del modelo

Para cada categoría y cada variable de promoción se calcula:

| Columna | Qué es |
|---|---|
| `beta` | coeficiente estimado |
| `ee` | error estándar agrupado por SKU y semana |
| `p_valor` | probabilidad de un coeficiente así de grande si el verdadero fuera 0; se usa la distribución $t$ con $G-1$ grados de libertad, donde $G$ es el número de clústeres de la dimensión más chica (semanas o SKUs) |
| `ic_inf`, `ic_sup` | intervalo de confianza de 95 % de `beta` |
| `efecto_pct` | $100\,(e^{\beta}-1)$: cambio % de la venta cuando la variable sube en 1 (de 0 a 1 en $D$; de 0 % a 100 % de los otros SKUs en promoción en $S$) |
| `efecto_10pp_pct` | solo para $S$: $100\,(e^{0{,}1\beta}-1)$, el cambio % si 10 puntos porcentuales más de los otros SKUs están en promoción |

In [5]:
filas = []
for categoria, modelo in modelos.items():
    beta = modelo.coef()[regresores_promocion]
    ee = modelo.se()[regresores_promocion]
    gl = modelo._df_t
    t_critico = stats.t.ppf(0.975, gl)
    filas.append(
        pd.DataFrame(
            {
                "categoria": categoria,
                "coeficiente": regresores_promocion,
                "beta": beta.values,
                "ee": ee.values,
                "p_valor": 2 * stats.t.sf(np.abs(beta / ee).values, gl),
                "ic_inf": (beta - t_critico * ee).values,
                "ic_sup": (beta + t_critico * ee).values,
            }
        )
    )
coeficientes = pd.concat(filas, ignore_index=True)

# Efectos en porcentaje
coeficientes["efecto_pct"] = 100 * np.expm1(coeficientes["beta"])
es_proporcion = coeficientes["coeficiente"].str.startswith("promo_")
coeficientes["efecto_10pp_pct"] = (100 * np.expm1(0.1 * coeficientes["beta"])).where(es_proporcion)

with pd.option_context("display.float_format", lambda x: f"{x:,.3f}"):
    print(coeficientes.set_index(["categoria", "coeficiente"]).to_string())
coeficientes.round(6).to_csv(os.path.join(ruta_tablas, "fase1_coeficientes.csv"), index=False)

                                               beta    ee  p_valor  ic_inf  ic_sup  efecto_pct  efecto_10pp_pct
categoria        coeficiente                                                                                   
BEVERAGES        onpromotion                  0.469 0.058    0.000   0.353   0.584      59.816              NaN
                 onpromotion_l1_3             0.238 0.062    0.000   0.115   0.361      26.923              NaN
                 onpromotion_l4_6            -0.105 0.063    0.100  -0.229   0.020      -9.923              NaN
                 promo_clase_otros           -0.306 0.114    0.009  -0.533  -0.078     -26.327           -3.009
                 promo_clase_otros_l1_3      -0.020 0.120    0.871  -0.259   0.220      -1.949           -0.197
                 promo_clase_otros_l4_6       0.079 0.101    0.441  -0.123   0.281       8.175            0.789
                 promo_cat_otras_clases      -0.633 0.361    0.084  -1.353   0.086     -46.926          

**Cómo leer la salida.** Cada bloque de nueve filas es el modelo de una categoría. Por ejemplo, en BEVERAGES `onpromotion` tiene $\beta = 0{,}469$: el día de la promoción la venta esperada del SKU es $e^{0{,}469} \approx 1{,}60$ veces la de un día comparable sin promoción (`efecto_pct` ≈ +60 %), con un intervalo que no incluye el 0 y $p<0{,}001$.

## Paso 6. Resumen por tipo de promoción

Los coeficientes del Paso 5 se reordenan en **tres tablas, una por tipo de promoción**:

- **Tabla 1. Efecto propio de la promoción** (`onpromotion`): variación % de la venta esperada del SKU cuando **él mismo** está en promoción.
- **Tabla 2. Otros SKU de la misma clase** (`promo_clase_otros`): variación % cuando 10 puntos porcentuales más de los otros SKUs de su clase están en promoción.
- **Tabla 3. Otras clases de la categoría** (`promo_cat_otras_clases`): variación % cuando 10 puntos porcentuales más de los SKUs de las otras clases de la categoría están en promoción.

Cada tabla tiene, para cada categoría, el efecto del **día de la promoción** y de los **rezagos de 1–3 y 4–6 días**, cada uno con su valor p. Un rezago negativo indicaría que las ventas caen en los días posteriores.

La Tabla 1 agrega el **efecto neto 7 días**, $100\,(e^{\beta_0+\beta_{\ell_{1\text{-}3}}+\beta_{\ell_{4\text{-}6}}}-1)$: el efecto de que el SKU haya estado en promoción **7 días seguidos**. Su intervalo de 95 % se calcula con el método delta (la varianza de la suma de los tres coeficientes es la suma de sus varianzas y covarianzas).

Los valores p usan los errores agrupados por SKU y semana, sin corrección por comparaciones múltiples.

In [6]:
def efecto_sostenido(modelo) -> tuple:
    """
    Calcula el efecto en % de una promoción sostenida 7 días y su intervalo de 95 % (método delta).

    :param modelo:  un modelo estimado con pyfixest.
    :return:        (efecto %, límite inferior %, límite superior %).
    """

    variables = ["onpromotion", "onpromotion_l1_3", "onpromotion_l4_6"]
    indices = [modelo._coefnames.index(v) for v in variables]
    suma = modelo.coef()[variables].sum()
    ee_suma = np.sqrt(modelo._vcov[np.ix_(indices, indices)].sum())
    t_critico = stats.t.ppf(0.975, modelo._df_t)
    return 100 * np.expm1(suma), 100 * np.expm1(suma - t_critico * ee_suma), 100 * np.expm1(suma + t_critico * ee_suma)


def formato_pct(x: float) -> str:
    """Escribe un efecto en % con signo y dos decimales (p. ej., +59.76 %)."""
    return f"{x:+.2f} %"


def formato_p(p: float) -> str:
    """Escribe un valor p con tres decimales, o <0.001."""
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def tabla_por_tipo(variable: str, columna_efecto: str) -> pd.DataFrame:
    """
    Arma la tabla de un tipo de promoción: efecto del día y de los rezagos 1-3 y 4-6, cada uno con su valor p.

    :param variable:        el nombre base de la variable (p. ej., 'onpromotion').
    :param columna_efecto:  'efecto_pct' (por unidad) o 'efecto_10pp_pct' (por +10 pp).
    :return:                un DataFrame con una fila por categoría.
    """

    c = coeficientes.set_index(["categoria", "coeficiente"])
    filas = []
    for categoria in categorias:
        fila = {"Categoría": categoria}
        for sufijo, nombre in [("", "Día de la promoción"), ("_l1_3", "Rezago 1-3"), ("_l4_6", "Rezago 4-6")]:
            r = c.loc[(categoria, variable + sufijo)]
            fila[nombre] = formato_pct(r[columna_efecto])
            fila[f"valor p ({nombre})"] = formato_p(r["p_valor"])
        filas.append(fila)
    return pd.DataFrame(filas)


# Tabla 1: efecto propio, con el efecto neto 7 días
tabla_1 = tabla_por_tipo("onpromotion", "efecto_pct")
neto = []
for categoria in categorias:
    efecto, inferior, superior = efecto_sostenido(modelos[categoria])
    neto.append(f"{formato_pct(efecto)} [{inferior:.2f}, {superior:.2f}]")
tabla_1["Efecto neto 7 días [IC 95 %]"] = neto

# Tablas 2 y 3: otros SKU de la clase y otras clases de la categoría, por +10 pp
tabla_2 = tabla_por_tipo("promo_clase_otros", "efecto_10pp_pct")
tabla_3 = tabla_por_tipo("promo_cat_otras_clases", "efecto_10pp_pct")

tablas = [
    ("Tabla 1. Efecto propio de la promoción (variación % de la venta esperada del SKU)", tabla_1, "fase1_tabla1_efecto_propio.csv"),
    ("Tabla 2. Otros SKU de la misma clase (variación % por +10 pp de la clase en promoción)", tabla_2, "fase1_tabla2_misma_clase.csv"),
    ("Tabla 3. Otras clases de la categoría (variación % por +10 pp en promoción)", tabla_3, "fase1_tabla3_otras_clases.csv"),
]
for titulo, tabla, archivo in tablas:
    # En pantalla, las columnas de valor p se muestran solo como "valor p"
    print(titulo)
    print(tabla.rename(columns=lambda x: "valor p" if x.startswith("valor p") else x).to_string(index=False))
    print()
    tabla.to_csv(os.path.join(ruta_tablas, archivo), index=False)

Tabla 1. Efecto propio de la promoción (variación % de la venta esperada del SKU)
       Categoría Día de la promoción valor p Rezago 1-3 valor p Rezago 4-6 valor p Efecto neto 7 días [IC 95 %]
       BEVERAGES            +59.82 %  <0.001   +26.92 %  <0.001    -9.92 %   0.100     +82.72 % [56.40, 113.46]
       GROCERY I            +60.43 %  <0.001   +29.65 %  <0.001    -1.08 %   0.801    +105.75 % [88.25, 124.87]
        CLEANING            +88.94 %  <0.001   +26.25 %   0.026    -2.40 %   0.727   +132.81 % [103.40, 166.47]
           DAIRY            +41.22 %  <0.001    +9.13 %  <0.001    +3.10 %   0.077      +58.88 % [49.72, 68.61]
    BREAD/BAKERY            +45.45 %  <0.001   +15.55 %   0.032   +10.35 %   0.192     +85.47 % [52.25, 125.92]
LIQUOR,WINE,BEER            +75.02 %  <0.001   +14.46 %   0.070    -0.80 %   0.938     +98.72 % [42.48, 177.16]

Tabla 2. Otros SKU de la misma clase (variación % por +10 pp de la clase en promoción)
       Categoría Día de la promoción valor p R

## Paso 7. Guardar el gráfico y las tablas como imágenes (PNG)

Este paso **no muestra nada en pantalla**: solo guarda, en `resultados/figuras/`, cuatro imágenes PNG (300 dpi) con el formato del póster, cada una generada por su propia celda:

1. `fase1_grafico_efectos.png`: gráfico de efectos por categoría, con tres paneles. **A.** Efecto propio el día de la promoción. **B.** Otros SKU de la misma clase (por +10 pp). **C.** Resto de la categoría (por +10 pp). Cada punto es el efecto estimado y la línea su intervalo de 95 %; punto lleno si $p<0{,}05$, vacío si no; en B y C, rojo es negativo (posible canibalización) y verde positivo.
2. `fase1_tabla1_efecto_propio.png`: Tabla 1, efecto propio, con el efecto neto 7 días.
3. `fase1_tabla2_misma_clase.png`: Tabla 2, otros SKU de la misma clase.
4. `fase1_tabla3_otras_clases.png`: Tabla 3, resto de la categoría (otras clases).

**Colores de las tablas** (según el valor p de cada coeficiente; la celda del efecto y la de su valor p llevan el mismo color): azul oscuro $p<0{,}01$, azul medio $p<0{,}05$, azul claro $p<0{,}10$ y blanco si no es significativo. La columna dorada es el efecto neto 7 días con su intervalo de 95 %. `DECIMALES` controla los decimales de los porcentajes (el póster usa 1); el color usa el valor p tal como se imprime (3 decimales).

In [7]:
# Preparación (sin salida): colores, funciones de dibujo y datos de las imágenes
import matplotlib
from matplotlib.patches import Rectangle

DECIMALES = 1   # decimales de los % en las imágenes

AZUL, ROJO, VERDE = "#1f5fa6", "#c0392b", "#27ae60"

def datos_panel(variable: str, por_10pp: bool) -> pd.DataFrame:
    """
    Junta, para una variable, el efecto en % y su intervalo de 95 % (en %) por categoría.

    :param variable:  nombre del coeficiente (p. ej., 'onpromotion').
    :param por_10pp:  True si el efecto se expresa por +10 pp (variables S); False si es por unidad.
    :return:          un DataFrame con una fila por categoría y columnas efecto, inferior, superior, p.
    """

    d = coeficientes[coeficientes["coeficiente"] == variable].set_index("categoria").loc[categorias]
    escala = 0.1 if por_10pp else 1.0
    return pd.DataFrame(
        {
            "efecto": 100 * np.expm1(escala * d["beta"]),
            "inferior": 100 * np.expm1(escala * d["ic_inf"]),
            "superior": 100 * np.expm1(escala * d["ic_sup"]),
            "p": d["p_valor"],
        }
    )


paneles = [
    ("A. Efecto propio el día de la promoción", datos_panel("onpromotion", False), "Cambio en la venta esperada del SKU (%)"),
    ("B. Otros SKUs de la misma clase", datos_panel("promo_clase_otros", True), "Cambio por +10 pp de la\nclase en promoción (%)"),
    ("C. Resto de la categoría", datos_panel("promo_cat_otras_clases", True), "Cambio por +10 pp de otras\nclases en promoción (%)"),
]

def dibujar_efectos(ejes, decimales: int = 2) -> None:
    """
    Dibuja en tres ejes los paneles A, B y C (efecto en % con su IC de 95 %, por categoría).

    :param ejes:       lista de 3 ejes de matplotlib, el primero con los nombres de categoría.
    :param decimales:  decimales de las etiquetas de porcentaje.
    """

    posiciones = np.arange(len(categorias))[::-1]   # la primera categoría arriba
    for indice, (eje, (titulo, d, etiqueta_x)) in enumerate(zip(ejes, paneles)):
        for posicion, categoria in zip(posiciones, categorias):
            fila = d.loc[categoria]
            if indice == 0:
                color = AZUL
            else:
                color = VERDE if fila["efecto"] > 0 else ROJO
            significativo = fila["p"] < 0.05
            eje.hlines(posicion, fila["inferior"], fila["superior"], color=color, linewidth=2)
            eje.plot(fila["efecto"], posicion, "o", markersize=8, color=color,
                     markerfacecolor=color if significativo else "white", markeredgewidth=1.8, zorder=3)
            eje.annotate(f"{fila['efecto']:+.{decimales}f} %".replace("-", "−"), (fila["efecto"], posicion),
                         textcoords="offset points", xytext=(0, -13), ha="center", fontsize=7.5, color="#444444")
        if indice > 0:
            eje.axvline(0, color="#555555", linewidth=1)
        eje.set_title(titulo, loc="left", fontsize=10.5, fontweight="bold")
        eje.set_xlabel(etiqueta_x, fontsize=8.5)
        eje.set_ylim(-0.75, len(categorias) - 0.4)
        eje.set_yticks(posiciones)
        eje.set_yticklabels(categorias, fontsize=9)
        eje.tick_params(axis="y", labelleft=(indice == 0))
        eje.grid(axis="x", color="#e5e5e5")
        eje.set_axisbelow(True)
        for lado in ["top", "right", "left"]:
            eje.spines[lado].set_visible(False)
        eje.tick_params(axis="y", length=0)


from matplotlib.patches import Rectangle

AZUL_OSC, AZUL_MED, AZUL_CLA = "#2b6a9e", "#8fb5db", "#dbe8f5"
NAVY, ORO_OSC, ORO_CLA = "#13213c", "#8a6914", "#f6d98a"
FONDO_CAT, BORDE = "#eef3f9", "#c5cfdb"

nombres_vista = {"BEVERAGES": "Beverages", "GROCERY I": "Grocery I", "CLEANING": "Cleaning", "DAIRY": "Dairy",
                 "BREAD/BAKERY": "Bread/Bakery", "LIQUOR,WINE,BEER": "Liquor, Wine, Beer"}

ANCHO_TAB, ALTO_CAB = 10.9, 0.75     # en pulgadas
ANCHO_FIG, MARGEN = 11.5, 0.3


def pct(x: float) -> str:
    """Escribe un efecto en % con signo (menos tipográfico) y DECIMALES decimales."""
    return f"{x:+.{DECIMALES}f} %".replace("-", "−")


def pval(p: float) -> str:
    """Escribe un valor p con tres decimales, o <0.001."""
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def estilo_p(p: float):
    """Devuelve (color de fondo, color de texto, peso) según la significancia del valor p mostrado."""
    p = round(p, 3)
    if p < 0.01:
        return AZUL_OSC, "white", "bold"
    if p < 0.05:
        return AZUL_MED, "#111111", "normal"
    if p < 0.10:
        return AZUL_CLA, "#111111", "normal"
    return "white", "#111111", "normal"


def celdas_tipo(variable: str, columna_efecto: str) -> dict:
    """Por categoría, la lista [(efecto %, valor p)] del día, rezago 1-3 y rezago 4-6."""
    c = coeficientes.set_index(["categoria", "coeficiente"])
    return {cat: [(c.loc[(cat, variable + sufijo), columna_efecto], c.loc[(cat, variable + sufijo), "p_valor"])
                  for sufijo in ["", "_l1_3", "_l4_6"]] for cat in categorias}


celdas_1 = celdas_tipo("onpromotion", "efecto_pct")
celdas_2 = celdas_tipo("promo_clase_otros", "efecto_10pp_pct")
celdas_3 = celdas_tipo("promo_cat_otras_clases", "efecto_10pp_pct")
netos = {cat: efecto_sostenido(modelos[cat]) for cat in categorias}   # (efecto %, inf %, sup %)


def dibujar_tabla(eje, celdas: dict, con_neto: bool, alto_fila: float) -> None:
    """Dibuja una tabla coloreada en un eje cuyas unidades son pulgadas."""
    anchos = np.array([2.0] + [1.55, 1.1] * 3 + ([2.0] if con_neto else []))
    anchos = anchos * ANCHO_TAB / anchos.sum()
    x = np.concatenate([[0], np.cumsum(anchos)])
    eje.set_xlim(0, ANCHO_TAB)
    eje.set_ylim(ALTO_CAB + alto_fila * len(categorias), 0)
    eje.axis("off")
    cabeceras = ["Categoría", "Día de la\npromoción", "valor p", "Rezago 1–3", "valor p", "Rezago 4–6", "valor p"]
    if con_neto:
        cabeceras.append("Efecto neto\n7 días")
    for j, texto in enumerate(cabeceras):
        fondo = ORO_OSC if (con_neto and j == 7) else NAVY
        eje.add_patch(Rectangle((x[j], 0), anchos[j], ALTO_CAB, facecolor=fondo, edgecolor="white", linewidth=1))
        eje.text(x[j] + anchos[j] / 2, ALTO_CAB / 2, texto, color="white", ha="center", va="center",
                 fontsize=9.5, fontweight="bold")
    for i, cat in enumerate(categorias):
        y0 = ALTO_CAB + i * alto_fila
        eje.add_patch(Rectangle((x[0], y0), anchos[0], alto_fila, facecolor=FONDO_CAT, edgecolor=BORDE, linewidth=0.8))
        eje.text(x[0] + anchos[0] / 2, y0 + alto_fila / 2, nombres_vista[cat], ha="center", va="center",
                 fontsize=9.5, fontweight="bold")
        for b, (efecto, p) in enumerate(celdas[cat]):
            fondo, color_texto, peso = estilo_p(p)
            for k, texto in enumerate([pct(efecto), pval(p)]):
                j = 1 + 2 * b + k
                eje.add_patch(Rectangle((x[j], y0), anchos[j], alto_fila, facecolor=fondo, edgecolor=BORDE, linewidth=0.8))
                eje.text(x[j] + anchos[j] / 2, y0 + alto_fila / 2, texto, color=color_texto, fontweight=peso,
                         ha="center", va="center", fontsize=9.5)
        if con_neto:
            e, inf, sup = netos[cat]
            eje.add_patch(Rectangle((x[7], y0), anchos[7], alto_fila, facecolor=ORO_CLA, edgecolor=BORDE, linewidth=0.8))
            eje.text(x[7] + anchos[7] / 2, y0 + alto_fila / 2, f"{pct(e)}\n[{inf:.{DECIMALES}f}, {sup:.{DECIMALES}f}]",
                     ha="center", va="center", fontsize=9, fontweight="bold")


def ejes_en(fig, W, H, x0, y_top, ancho, alto, **kw):
    """Crea un eje con esquina superior izquierda en (x0, y_top) y tamaño (ancho, alto), todo en pulgadas."""
    return fig.add_axes([x0 / W, 1 - (y_top + alto) / H, ancho / W, alto / H], **kw)


def tabla_en_figura(fig, W, H, x0, y_top, titulo, celdas, con_neto, alto_fila) -> float:
    """Pone título y tabla en la figura. Devuelve la coordenada y donde termina."""
    fig.text(x0 / W, 1 - (y_top + 0.15) / H, titulo, fontsize=11.5, fontweight="bold", va="center", ha="left")
    alto = ALTO_CAB + alto_fila * len(categorias)
    dibujar_tabla(ejes_en(fig, W, H, x0, y_top + 0.4, ANCHO_TAB, alto), celdas, con_neto, alto_fila)
    return y_top + 0.4 + alto


def leyenda_en_figura(fig, W, H, x0, y_top) -> float:
    """Leyenda de colores y nota al pie de la Tabla 1. Devuelve la coordenada y donde termina."""
    eje = ejes_en(fig, W, H, x0, y_top + 0.1, ANCHO_TAB, 0.3)
    eje.set_xlim(0, ANCHO_TAB)
    eje.set_ylim(0.3, 0)
    eje.axis("off")
    items = [(AZUL_OSC, "p < 0.01", 0.0), (AZUL_MED, "p < 0.05", 1.8), (AZUL_CLA, "p < 0.10", 3.6),
             ("white", "no significativo", 5.4), (ORO_CLA, "efecto neto (IC 95 %)", 7.8)]
    for fondo, texto, xi in items:
        eje.add_patch(Rectangle((xi, 0.05), 0.35, 0.2, facecolor=fondo, edgecolor=BORDE, linewidth=0.8))
        eje.text(xi + 0.45, 0.15, texto, va="center", fontsize=9)
    fig.text(x0 / W, 1 - (y_top + 0.62) / H,
             "Efecto neto 7 días = e^(β₀+β₁₋₃+β₄₋₆) − 1: SKU en promoción durante 7 días seguidos; IC por método delta. "
             "Valor p con EE agrupados por SKU y semana, sin corrección múltiple.",
             fontsize=7.5, color="#555555", va="center")
    return y_top + 0.8


def guardar(fig, nombre: str) -> None:
    """Guarda la figura como PNG (300 dpi) en resultados/figuras y la cierra, sin mostrarla."""
    ruta = os.path.join(ruta_figuras, nombre + ".png")
    fig.savefig(ruta, dpi=300)
    plt.close(fig)
    print("Guardada:", ruta)


def guardar_tabla(titulo: str, celdas: dict, con_neto: bool, alto_fila: float, nombre: str) -> None:
    """Dibuja una tabla coloreada (con la leyenda si lleva efecto neto) y la guarda como PNG."""
    H = 0.2 + 0.4 + ALTO_CAB + alto_fila * len(categorias) + (0.8 if con_neto else 0) + 0.2
    fig_t = plt.figure(figsize=(ANCHO_FIG, H), facecolor="white")
    y = tabla_en_figura(fig_t, ANCHO_FIG, H, MARGEN, 0.2, titulo, celdas, con_neto, alto_fila)
    if con_neto:
        leyenda_en_figura(fig_t, ANCHO_FIG, H, MARGEN, y)
    guardar(fig_t, nombre)

### 7.1 Gráfico de efectos por categoría

In [8]:
H = 5.0
fig = plt.figure(figsize=(ANCHO_FIG, H), facecolor="white")
fig.text(MARGEN / ANCHO_FIG, 1 - 0.35 / H,
         "Efectos asociados a la promoción por categoría (PPML, IC 95 %, EE agrupados por SKU y semana)", fontsize=11, va="center")
x_ini, ancho_p, sep = 1.9, 2.55, 0.45
ejes_g = []
for k in range(3):
    ejes_g.append(ejes_en(fig, ANCHO_FIG, H, x_ini + k * (ancho_p + sep), 1.1, ancho_p, 2.8,
                          **({"sharey": ejes_g[0]} if k else {})))
dibujar_efectos(ejes_g, decimales=DECIMALES)
fig.text(MARGEN / ANCHO_FIG, 1 - 4.8 / H,
         "Punto lleno: p < 0,05. Punto vacío: no significativo al 5 %. Efecto = 100(e^β − 1). Tienda 44, 2016–2017; 532 días.",
         fontsize=8, color="#888888", va="center")
guardar(fig, "fase1_grafico_efectos")

Guardada: resultados/figuras/fase1_grafico_efectos.png


### 7.2 Tabla 1. Efecto propio de la promoción

In [9]:
guardar_tabla("Tabla 1. Efecto propio de la promoción (variación % de la venta esperada del SKU)",
              celdas_1, True, 0.65, "fase1_tabla1_efecto_propio")

Guardada: resultados/figuras/fase1_tabla1_efecto_propio.png


### 7.3 Tabla 2. Otros SKU de la misma clase

In [10]:
guardar_tabla("Tabla 2. Otros SKU de la misma clase (variación % por +10 pp de la clase en promoción)",
              celdas_2, False, 0.5, "fase1_tabla2_misma_clase")

Guardada: resultados/figuras/fase1_tabla2_misma_clase.png


### 7.4 Tabla 3. Resto de la categoría (otras clases)

In [11]:
guardar_tabla("Tabla 3. Otras clases de la categoría (variación % por +10 pp en promoción)",
              celdas_3, False, 0.5, "fase1_tabla3_otras_clases")

Guardada: resultados/figuras/fase1_tabla3_otras_clases.png


## Paso 8. Lectura de los resultados

**Tabla 1 · Efecto propio el día de la promoción.** Es positivo y significativo al 1 % en las seis categorías: entre **+41 %** (DAIRY) y **+89 %** (CLEANING). Los intervalos de varias categorías se superponen, así que el orden entre ellas es solo indicativo.

**Tabla 1 · ¿Caen las ventas después de la promoción?** No se observa una caída: el rezago de 1–3 días es positivo en las seis categorías (significativo al 5 % en cinco; en LIQUOR,WINE,BEER solo al 10 %), y el de 4–6 días no es significativo al 5 % en ninguna. Esto **no prueba** que no haya desplazamiento intertemporal: las campañas duran varios días, así que el rezago de 1–3 días mezcla el efecto de llevar varios días en promoción con lo que pasa cuando la promoción termina. El efecto neto 7 días va de +59 % (DAIRY) a +133 % (CLEANING).

**Tabla 2 · Otros SKU de la misma clase.** En BEVERAGES, GROCERY I, CLEANING y DAIRY el coeficiente es negativo y significativo al 5 %: cuando 10 puntos porcentuales más de los otros SKUs de la clase están en promoción, cada SKU vende entre 1 % y 3 % menos. Es consistente con **canibalización entre productos cercanos**, pero el efecto es pequeño. En LIQUOR,WINE,BEER sale positivo; es un resultado frágil (pocos SKUs y muchas clases con un solo SKU) y no conviene interpretarlo.

**Tabla 3 · Otras clases de la categoría.** Es negativo en las seis categorías, pero solo significativo al 5 % en BREAD/BAKERY: con estos datos no se puede afirmar un efecto sistemático.

**Alcance.** Son **efectos asociados**, condicionales al calendario y a los efectos fijos, para la tienda 44 en 2016–2017 y para estas seis categorías. No se extienden a las categorías con programa semanal.